# Notebook 11 — Baseline Target Audience (BTA) Cards

## Purpose

This notebook converts the previously identified baseline clusters into **Baseline Target Audience (BTA) cards** that are readable by analysts and usable by downstream AI systems.

The goal is to transform cluster-level outputs into a structured **baseline audience intelligence layer** for Market Kinetics.

These BTA cards do **not** represent client-specific customer segments. Instead, they describe broad, society-level audience archetypes derived from the integrated baseline data stack:

- **ACS (American Community Survey)** for structural and demographic characteristics
- **GSS (General Social Survey)** for psychological and attitudinal signals
- **NPORS / Pew datasets** for media behavior indicators

This baseline segmentation provides a **societal reference layer** that Market Kinetics can later combine with proprietary customer and behavioral data to identify more refined segments and, ultimately, generate objective-specific **Target Audience Analysis Worksheets (TAAWs)**.

---

## Notebook outputs

This notebook performs the following steps:

1. Converts cluster-level profiles into one row per **Baseline Target Audience (BTA)**
2. Preserves dominant structural traits derived from ACS
3. Incorporates psychological signals inferred from the GSS projection layer
4. Incorporates media behavior signals derived from NPORS / Pew datasets
5. Generates analyst-readable summaries describing structural, psychological, and media characteristics
6. Produces a compact **RAG-ready representation** of each BTA for downstream retrieval and reasoning workflows

---

## Expected deliverables

Primary dataset:

- `mk_bta_baseline.parquet`

Optional export formats:

- `mk_bta_baseline.csv` — tabular export for inspection and sharing
- `mk_bta_rag_corpus.jsonl` — retrieval corpus designed for embedding and vector search pipelines

---

## Design principle

Each BTA card is designed to function simultaneously as:

- a **human-readable audience intelligence brief**
- a **machine-readable retrieval object** suitable for downstream AI workflows

This structure enables Market Kinetics to support both **analyst interpretation** and **AI-assisted audience selection**, campaign planning, and future **Target Audience Analysis Worksheet (TAAW)** generation once objective-specific data is introduced.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

In [2]:
# Paths
PROJECT_ROOT  = Path().resolve().parent

DATA_DIR      = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "societal_processed"
MODELS_DIR    = DATA_DIR / "societal_models"
INTERIM_DIR   = DATA_DIR / "societal_interim"

OUTPUT_DIR    = PROCESSED_DIR / "bta_cards"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Output dir   :", OUTPUT_DIR)

Project root : /Users/marcomagnolo/Projects/Market_Kinetics
Output dir   : /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_processed/bta_cards


In [3]:
# loading cluster profiles
cluster_profiles_path = PROCESSED_DIR / "10_mk_cluster_profiles.parquet"

df_cluster_profiles = pd.read_parquet(cluster_profiles_path)

print("Loaded:", cluster_profiles_path)
print("Shape:", df_cluster_profiles.shape)

df_cluster_profiles.head()

Loaded: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_processed/10_mk_cluster_profiles.parquet
Shape: (7, 4)


,cluster,population_share,psych_signals,media_signals
0,0,0.055954,"[media_engagement: low, media_engagement: high...","[media_usage: tiktok, media_usage: whatsapp, m..."
1,1,0.123532,"[media_engagement: low, religiosity: none, ide...","[media_usage: internet_frequency, media_usage:..."
2,2,0.183602,"[party_alignment: republican, media_engagement...","[media_usage: internet_frequency, media_usage:..."
3,3,0.190217,[media_engagement: low],"[media_usage: internet_frequency, media_usage:..."
4,4,0.101945,"[religiosity: none, media_engagement: low, rel...","[media_usage: internet_frequency, media_usage:..."


In [4]:
# loading structural archetypes
archetypes_path = PROCESSED_DIR / "06b_mk_us_structural_archetypes_v2.csv"

df_archetypes = pd.read_csv(archetypes_path)
df_archetypes.head(7)

,cluster_id,archetype_name,population_pct_total,population_pct_adult,dominant_emp_tier,dominant_hhincome_tier,dominant_tenure,dominant_edu_tier,dominant_mar_tier,dominant_age_bin,dominant_race_eth,dominant_sex_label
0,0,Young Diverse Working Households,0.056,0.056,Employed,100-199k,Owner,HS_or_less,Married,35-44,Hispanic,Male
1,1,Older White Renters,0.124,0.124,Retired,50-99k,Renter,HS_or_less,Married,65+,White_NH,Female
2,2,Educated Affluent Homeowners,0.184,0.184,Employed,100-199k,Owner,Bachelor,Married,55-64,White_NH,Female
3,3,Mainstream Working Families,0.190,0.190,Employed,50-99k,Owner,HS_or_less,Married,35-44,White_NH,Female
4,4,Fixed-Income Late-Life Adults,0.102,0.102,Retired,0-19k,No_Rent,HS_or_less,Previously_Married,65+,White_NH,Female
5,5,Young Non-Owning Working Adults,0.204,0.204,Employed,20-49k,No_Rent,HS_or_less,Never_Married,25-34,White_NH,Male
6,6,White Male Multi-Generation Households,0.140,0.140,Employed,100-199k,Owner,HS_or_less,Never_Married,18-24,White_NH,Male


In [5]:
df_cluster_profiles.head()

,cluster,population_share,psych_signals,media_signals
0,0,0.055954,"[media_engagement: low, media_engagement: high...","[media_usage: tiktok, media_usage: whatsapp, m..."
1,1,0.123532,"[media_engagement: low, religiosity: none, ide...","[media_usage: internet_frequency, media_usage:..."
2,2,0.183602,"[party_alignment: republican, media_engagement...","[media_usage: internet_frequency, media_usage:..."
3,3,0.190217,[media_engagement: low],"[media_usage: internet_frequency, media_usage:..."
4,4,0.101945,"[religiosity: none, media_engagement: low, rel...","[media_usage: internet_frequency, media_usage:..."


In [6]:
print(df_cluster_profiles.loc[4, "psych_signals"])

['religiosity: none' 'media_engagement: low' 'religiosity: high']


In [7]:
# Build BTA base table
df_bta = pd.DataFrame({
    "segment_id":             df_cluster_profiles["cluster"].apply(lambda x: f"BTA_{int(x):02d}"),
    "cluster_id":             df_cluster_profiles["cluster"],
    "archetype_name":         None,
    "baseline_layer_version": "v1",
    "population_share":       df_cluster_profiles["population_share"],
    "population_rank":        None,

    "dominant_age_bin":       None,
    "dominant_sex_label":     None,
    "dominant_race_eth":      None,
    "dominant_edu_tier":      None,
    "dominant_emp_tier":      None,
    "dominant_income_tier":   None,
    "dominant_mar_tier":      None,
    "dominant_tenure":        None,
    "structural_profile":     None,

    "psych_signals":          df_cluster_profiles["psych_signals"],
    "psych_summary":          None,
    "media_signals":          df_cluster_profiles["media_signals"],
    "media_summary":          None,

    "motivational_drivers":   None,
    "key_barriers":           None,
    "trust_cues":             None,
    "channel_implications":   None,
    "messaging_implications": None,
    "offer_implications":     None,
    "susceptibility_notes":   None,

    "narrative_summary":      None,
    "tags":                   None,
    "source_layers":          None,
    "rag_text":               None,
})

df_bta.head()

,segment_id,cluster_id,archetype_name,baseline_layer_version,population_share,population_rank,dominant_age_bin,dominant_sex_label,dominant_race_eth,dominant_edu_tier,...,key_barriers,trust_cues,channel_implications,messaging_implications,offer_implications,susceptibility_notes,narrative_summary,tags,source_layers,rag_text
0,BTA_00,0,None,v1,0.055954,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,BTA_01,1,None,v1,0.123532,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,BTA_02,2,None,v1,0.183602,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,BTA_03,3,None,v1,0.190217,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,BTA_04,4,None,v1,0.101945,None,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [8]:
# Merge archetypes data into BTA dataframe

df_bta = df_bta.merge(
    df_archetypes,
    on="cluster_id",
    how="left"
)

df_bta.head()

,segment_id,cluster_id,archetype_name_x,baseline_layer_version,population_share,population_rank,dominant_age_bin_x,dominant_sex_label_x,dominant_race_eth_x,dominant_edu_tier_x,...,population_pct_total,population_pct_adult,dominant_emp_tier_y,dominant_hhincome_tier,dominant_tenure_y,dominant_edu_tier_y,dominant_mar_tier_y,dominant_age_bin_y,dominant_race_eth_y,dominant_sex_label_y
0,BTA_00,0,None,v1,0.055954,None,None,None,None,None,...,0.056,0.056,Employed,100-199k,Owner,HS_or_less,Married,35-44,Hispanic,Male
1,BTA_01,1,None,v1,0.123532,None,None,None,None,None,...,0.124,0.124,Retired,50-99k,Renter,HS_or_less,Married,65+,White_NH,Female
2,BTA_02,2,None,v1,0.183602,None,None,None,None,None,...,0.184,0.184,Employed,100-199k,Owner,Bachelor,Married,55-64,White_NH,Female
3,BTA_03,3,None,v1,0.190217,None,None,None,None,None,...,0.190,0.190,Employed,50-99k,Owner,HS_or_less,Married,35-44,White_NH,Female
4,BTA_04,4,None,v1,0.101945,None,None,None,None,None,...,0.102,0.102,Retired,0-19k,No_Rent,HS_or_less,Previously_Married,65+,White_NH,Female


In [9]:
df_bta.columns

Index(['segment_id', 'cluster_id', 'archetype_name_x',
       'baseline_layer_version', 'population_share', 'population_rank',
       'dominant_age_bin_x', 'dominant_sex_label_x', 'dominant_race_eth_x',
       'dominant_edu_tier_x', 'dominant_emp_tier_x', 'dominant_income_tier',
       'dominant_mar_tier_x', 'dominant_tenure_x', 'structural_profile',
       'psych_signals', 'psych_summary', 'media_signals', 'media_summary',
       'motivational_drivers', 'key_barriers', 'trust_cues',
       'channel_implications', 'messaging_implications', 'offer_implications',
       'susceptibility_notes', 'narrative_summary', 'tags', 'source_layers',
       'rag_text', 'archetype_name_y', 'population_pct_total',
       'population_pct_adult', 'dominant_emp_tier_y', 'dominant_hhincome_tier',
       'dominant_tenure_y', 'dominant_edu_tier_y', 'dominant_mar_tier_y',
       'dominant_age_bin_y', 'dominant_race_eth_y', 'dominant_sex_label_y'],
      dtype='str')

In [10]:
# renaming columns
df_bta = df_bta.rename(columns={
    "population_pct_adult": "population_adult_share",
    "dominant_hhincome_tier": "dominant_household_income_tier"
})

In [11]:
# drop redundant columns from archetypes
df_bta = df_bta.drop(columns=[
    "population_pct_total",
    "dominant_income_tier",
])

df_bta.head()

,segment_id,cluster_id,archetype_name_x,baseline_layer_version,population_share,population_rank,dominant_age_bin_x,dominant_sex_label_x,dominant_race_eth_x,dominant_edu_tier_x,...,archetype_name_y,population_adult_share,dominant_emp_tier_y,dominant_household_income_tier,dominant_tenure_y,dominant_edu_tier_y,dominant_mar_tier_y,dominant_age_bin_y,dominant_race_eth_y,dominant_sex_label_y
0,BTA_00,0,None,v1,0.055954,None,None,None,None,None,...,Young Diverse Working Households,0.056,Employed,100-199k,Owner,HS_or_less,Married,35-44,Hispanic,Male
1,BTA_01,1,None,v1,0.123532,None,None,None,None,None,...,Older White Renters,0.124,Retired,50-99k,Renter,HS_or_less,Married,65+,White_NH,Female
2,BTA_02,2,None,v1,0.183602,None,None,None,None,None,...,Educated Affluent Homeowners,0.184,Employed,100-199k,Owner,Bachelor,Married,55-64,White_NH,Female
3,BTA_03,3,None,v1,0.190217,None,None,None,None,None,...,Mainstream Working Families,0.190,Employed,50-99k,Owner,HS_or_less,Married,35-44,White_NH,Female
4,BTA_04,4,None,v1,0.101945,None,None,None,None,None,...,Fixed-Income Late-Life Adults,0.102,Retired,0-19k,No_Rent,HS_or_less,Previously_Married,65+,White_NH,Female


In [12]:
# Clean merged BTA columns after archetype merge
df_bta = df_bta.drop(columns=[
    "archetype_name_x",
    "dominant_age_bin_x",
    "dominant_sex_label_x",
    "dominant_race_eth_x",
    "dominant_edu_tier_x",
    "dominant_emp_tier_x",
    "dominant_mar_tier_x",
    "dominant_tenure_x",
])

df_bta = df_bta.rename(columns={
    "archetype_name_y":      "archetype_name",
    "dominant_age_bin_y":    "dominant_age_bin",
    "dominant_sex_label_y":  "dominant_sex_label",
    "dominant_race_eth_y":   "dominant_race_eth",
    "dominant_edu_tier_y":   "dominant_edu_tier",
    "dominant_emp_tier_y":   "dominant_emp_tier",
    "dominant_mar_tier_y":   "dominant_mar_tier",
    "dominant_tenure_y":     "dominant_tenure",
})

print(df_bta.columns.tolist())
df_bta.head()

['segment_id', 'cluster_id', 'baseline_layer_version', 'population_share', 'population_rank', 'structural_profile', 'psych_signals', 'psych_summary', 'media_signals', 'media_summary', 'motivational_drivers', 'key_barriers', 'trust_cues', 'channel_implications', 'messaging_implications', 'offer_implications', 'susceptibility_notes', 'narrative_summary', 'tags', 'source_layers', 'rag_text', 'archetype_name', 'population_adult_share', 'dominant_emp_tier', 'dominant_household_income_tier', 'dominant_tenure', 'dominant_edu_tier', 'dominant_mar_tier', 'dominant_age_bin', 'dominant_race_eth', 'dominant_sex_label']


,segment_id,cluster_id,baseline_layer_version,population_share,population_rank,structural_profile,psych_signals,psych_summary,media_signals,media_summary,...,archetype_name,population_adult_share,dominant_emp_tier,dominant_household_income_tier,dominant_tenure,dominant_edu_tier,dominant_mar_tier,dominant_age_bin,dominant_race_eth,dominant_sex_label
0,BTA_00,0,v1,0.055954,None,None,"[media_engagement: low, media_engagement: high...",None,"[media_usage: tiktok, media_usage: whatsapp, m...",None,...,Young Diverse Working Households,0.056,Employed,100-199k,Owner,HS_or_less,Married,35-44,Hispanic,Male
1,BTA_01,1,v1,0.123532,None,None,"[media_engagement: low, religiosity: none, ide...",None,"[media_usage: internet_frequency, media_usage:...",None,...,Older White Renters,0.124,Retired,50-99k,Renter,HS_or_less,Married,65+,White_NH,Female
2,BTA_02,2,v1,0.183602,None,None,"[party_alignment: republican, media_engagement...",None,"[media_usage: internet_frequency, media_usage:...",None,...,Educated Affluent Homeowners,0.184,Employed,100-199k,Owner,Bachelor,Married,55-64,White_NH,Female
3,BTA_03,3,v1,0.190217,None,None,[media_engagement: low],None,"[media_usage: internet_frequency, media_usage:...",None,...,Mainstream Working Families,0.190,Employed,50-99k,Owner,HS_or_less,Married,35-44,White_NH,Female
4,BTA_04,4,v1,0.101945,None,None,"[religiosity: none, media_engagement: low, rel...",None,"[media_usage: internet_frequency, media_usage:...",None,...,Fixed-Income Late-Life Adults,0.102,Retired,0-19k,No_Rent,HS_or_less,Previously_Married,65+,White_NH,Female


In [13]:
df_bta.columns

Index(['segment_id', 'cluster_id', 'baseline_layer_version',
       'population_share', 'population_rank', 'structural_profile',
       'psych_signals', 'psych_summary', 'media_signals', 'media_summary',
       'motivational_drivers', 'key_barriers', 'trust_cues',
       'channel_implications', 'messaging_implications', 'offer_implications',
       'susceptibility_notes', 'narrative_summary', 'tags', 'source_layers',
       'rag_text', 'archetype_name', 'population_adult_share',
       'dominant_emp_tier', 'dominant_household_income_tier',
       'dominant_tenure', 'dominant_edu_tier', 'dominant_mar_tier',
       'dominant_age_bin', 'dominant_race_eth', 'dominant_sex_label'],
      dtype='str')

In [14]:
df_bta.head()

,segment_id,cluster_id,baseline_layer_version,population_share,population_rank,structural_profile,psych_signals,psych_summary,media_signals,media_summary,...,archetype_name,population_adult_share,dominant_emp_tier,dominant_household_income_tier,dominant_tenure,dominant_edu_tier,dominant_mar_tier,dominant_age_bin,dominant_race_eth,dominant_sex_label
0,BTA_00,0,v1,0.055954,None,None,"[media_engagement: low, media_engagement: high...",None,"[media_usage: tiktok, media_usage: whatsapp, m...",None,...,Young Diverse Working Households,0.056,Employed,100-199k,Owner,HS_or_less,Married,35-44,Hispanic,Male
1,BTA_01,1,v1,0.123532,None,None,"[media_engagement: low, religiosity: none, ide...",None,"[media_usage: internet_frequency, media_usage:...",None,...,Older White Renters,0.124,Retired,50-99k,Renter,HS_or_less,Married,65+,White_NH,Female
2,BTA_02,2,v1,0.183602,None,None,"[party_alignment: republican, media_engagement...",None,"[media_usage: internet_frequency, media_usage:...",None,...,Educated Affluent Homeowners,0.184,Employed,100-199k,Owner,Bachelor,Married,55-64,White_NH,Female
3,BTA_03,3,v1,0.190217,None,None,[media_engagement: low],None,"[media_usage: internet_frequency, media_usage:...",None,...,Mainstream Working Families,0.190,Employed,50-99k,Owner,HS_or_less,Married,35-44,White_NH,Female
4,BTA_04,4,v1,0.101945,None,None,"[religiosity: none, media_engagement: low, rel...",None,"[media_usage: internet_frequency, media_usage:...",None,...,Fixed-Income Late-Life Adults,0.102,Retired,0-19k,No_Rent,HS_or_less,Previously_Married,65+,White_NH,Female


In [15]:
bta_columns = [
    "segment_id",
    "cluster_id",
    "archetype_name",
    "baseline_layer_version",
    "population_share",
    "population_adult_share",
    "population_rank",

    "dominant_age_bin",
    "dominant_sex_label",
    "dominant_race_eth",
    "dominant_edu_tier",
    "dominant_emp_tier",
    "dominant_household_income_tier",
    "dominant_mar_tier",
    "dominant_tenure",
    "structural_profile",

    "psych_signals",
    "psych_summary",
    "media_signals",
    "media_summary",

    "motivational_drivers",
    "key_barriers",
    "trust_cues",
    "channel_implications",
    "messaging_implications",
    "offer_implications",
    "susceptibility_notes",

    "narrative_summary",
    "tags",
    "source_layers",
    "rag_text",
]

In [16]:
df_bta = df_bta[bta_columns]

df_bta.head()

,segment_id,cluster_id,archetype_name,baseline_layer_version,population_share,population_adult_share,population_rank,dominant_age_bin,dominant_sex_label,dominant_race_eth,...,key_barriers,trust_cues,channel_implications,messaging_implications,offer_implications,susceptibility_notes,narrative_summary,tags,source_layers,rag_text
0,BTA_00,0,Young Diverse Working Households,v1,0.055954,0.056,None,35-44,Male,Hispanic,...,None,None,None,None,None,None,None,None,None,None
1,BTA_01,1,Older White Renters,v1,0.123532,0.124,None,65+,Female,White_NH,...,None,None,None,None,None,None,None,None,None,None
2,BTA_02,2,Educated Affluent Homeowners,v1,0.183602,0.184,None,55-64,Female,White_NH,...,None,None,None,None,None,None,None,None,None,None
3,BTA_03,3,Mainstream Working Families,v1,0.190217,0.190,None,35-44,Female,White_NH,...,None,None,None,None,None,None,None,None,None,None
4,BTA_04,4,Fixed-Income Late-Life Adults,v1,0.101945,0.102,None,65+,Female,White_NH,...,None,None,None,None,None,None,None,None,None,None


In [17]:
# Clean race/ethnicity labels by removing "_NH"
# Example: "White_NH" -> "White"

df_bta["dominant_race_eth"] = (
    df_bta["dominant_race_eth"]
    .astype(str)
    .str.replace("_NH", "", regex=False)
)

In [18]:
df_bta[["segment_id", "dominant_race_eth"]]

,segment_id,dominant_race_eth
0,BTA_00,Hispanic
1,BTA_01,White
2,BTA_02,White
3,BTA_03,White
4,BTA_04,White
5,BTA_05,White
6,BTA_06,White


### BTA Card Initialization — Structural Backbone Integration

In this phase, the **initial BTA card structure** was created and
populated with the baseline information available at the cluster level.

First, the cluster-level intelligence dataset
(`10_mk_cluster_profiles.parquet`) was used to generate the **BTA
skeleton**, establishing one row per segment and preserving key
baseline attributes such as:

- `segment_id`
- `cluster_id`
- `population_share`
- `psych_signals`
- `media_signals`

Next, the **structural archetype layer** (`df_archetypes`) was merged
into the BTA table. This step added the dominant demographic and
socioeconomic traits associated with each cluster, including dominant
age group, sex, race/ethnicity, education level, employment status,
household income tier, marital status, and housing tenure.

Redundant columns created during the merge were cleaned, and column
names were standardized to match the BTA schema. The final dataframe
was reordered according to the defined BTA card structure.

At this stage, the BTA table represents a **structural and behavioral
backbone** for each baseline segment of the U.S. population.
Interpretation and narrative fields remain empty and will be generated
in subsequent steps.

### Next Step

The next phase generates the **`structural_profile`** field, which
converts the dominant structural attributes of each segment into a
concise, human-readable description. This serves as the first narrative
component of each BTA card and supports the construction of
`narrative_summary` and `rag_text`.

In [19]:
# building structural profile narrative based on dominant demographic attributes
def build_structural_profile(row):

    parts = []

    tenure_labels = {
        "Owner":          "homeowners",
        "Renter":         "renters",
        "No_Rent":        "non-paying occupants",
        "Group_Quarters": "group quarters residents",
        "Other":          "other housing arrangements",
    }

    if pd.notna(row["dominant_age_bin"]):
        parts.append(f"primarily aged {row['dominant_age_bin']}")

    if pd.notna(row["dominant_emp_tier"]):
        parts.append(f"{row['dominant_emp_tier'].lower()}")

    if pd.notna(row["dominant_mar_tier"]):
        parts.append(f"{row['dominant_mar_tier'].replace('_', ' ').lower()}")

    if pd.notna(row["dominant_edu_tier"]):
        parts.append(f"with {row['dominant_edu_tier'].replace('_', ' ').lower()} education")

    if pd.notna(row["dominant_household_income_tier"]):
        parts.append(f"household income typically in the {row['dominant_household_income_tier']} range")

    if pd.notna(row["dominant_tenure"]):
        label = tenure_labels.get(row["dominant_tenure"], row["dominant_tenure"].lower())
        parts.append(f"mostly {label}")

    if pd.notna(row["dominant_race_eth"]):
        parts.append(f"predominantly {row['dominant_race_eth'].replace('_', ' ')}")

    return "Segment composed of individuals " + ", ".join(parts) + "."

In [20]:
# apply
df_bta["structural_profile"] = df_bta.apply(build_structural_profile, axis=1)

df_bta[[
    "segment_id",
    "archetype_name",
    "structural_profile"
]].head()

,segment_id,archetype_name,structural_profile
0,BTA_00,Young Diverse Working Households,Segment composed of individuals primarily aged...
1,BTA_01,Older White Renters,Segment composed of individuals primarily aged...
2,BTA_02,Educated Affluent Homeowners,Segment composed of individuals primarily aged...
3,BTA_03,Mainstream Working Families,Segment composed of individuals primarily aged...
4,BTA_04,Fixed-Income Late-Life Adults,Segment composed of individuals primarily aged...


In [21]:
df_bta.columns

Index(['segment_id', 'cluster_id', 'archetype_name', 'baseline_layer_version',
       'population_share', 'population_adult_share', 'population_rank',
       'dominant_age_bin', 'dominant_sex_label', 'dominant_race_eth',
       'dominant_edu_tier', 'dominant_emp_tier',
       'dominant_household_income_tier', 'dominant_mar_tier',
       'dominant_tenure', 'structural_profile', 'psych_signals',
       'psych_summary', 'media_signals', 'media_summary',
       'motivational_drivers', 'key_barriers', 'trust_cues',
       'channel_implications', 'messaging_implications', 'offer_implications',
       'susceptibility_notes', 'narrative_summary', 'tags', 'source_layers',
       'rag_text'],
      dtype='str')

### CLAUDE INTEGRATION
Using claude to generate pyschological, media and narrative summaries for the clusters.

In [22]:
import anthropic
import os
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")

_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

In [23]:
# PROMPT FOR PSYCH SIGNALS SUMMARY
PSYCH_PROMPT = """
You are an audience intelligence analyst working on a baseline segmentation of the United States population.

Your task is to interpret the psychological signals associated with one audience segment and produce a short analytical summary.

The signals represent statistically meaningful psychological or attitudinal indicators associated with the segment relative to the broader population.

Your goal is to translate these signals into a clear, human-readable description of the segment’s likely attitudinal tendencies.

Guidelines:

• Write 2–3 concise sentences.
• Use neutral analytical language (not marketing language).
• Use cautious phrasing such as "appears", "suggests", "may indicate", or "is associated with".
• Preserve all important signals, especially ideological or political cues if present.
• If signals appear mixed or contradictory, acknowledge the ambiguity rather than collapsing them.
• Do not invent traits not supported by the signals.
• Do not provide recommendations or messaging.

Segment information:

Segment name:
{archetype_name}

Psychological signals detected in the cluster:
{psych_signals}

Task:

Write a concise analytical description of the psychological tendencies of this audience segment based strictly on the signals provided.

Output only the summary text.
Do not include headings, bullet points, or explanations.
"""

In [24]:
# function to generate psych summary using the prompt and the model
def normalize_signals(x):
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    if isinstance(x, np.ndarray):
        x = x.tolist()
    if isinstance(x, list):
        cleaned = [str(v).strip() for v in x if str(v).strip()]
        return ", ".join(cleaned) if cleaned else None
    x = str(x).strip()
    return x if x else None

In [25]:
# iterate over BTA dataframe and generate psych summary for each row
for i, row in df_bta.iterrows():

    psych_text = normalize_signals(row["psych_signals"])

    if psych_text is None:
        df_bta.loc[i, "psych_summary"] = None
        continue

    prompt = PSYCH_PROMPT.format(
        archetype_name=row["archetype_name"],
        psych_signals=psych_text
    )

    response = _client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )

    df_bta.loc[i, "psych_summary"] = response.content[0].text.strip()

In [26]:
# Display the relevant columns to review the generated summaries
df_bta[["segment_id", "archetype_name", "psych_signals", "psych_summary"]]

,segment_id,archetype_name,psych_signals,psych_summary
0,BTA_00,Young Diverse Working Households,"[media_engagement: low, media_engagement: high...",This segment shows mixed signals around media ...
1,BTA_01,Older White Renters,"[media_engagement: low, religiosity: none, ide...",This segment appears to hold conservative ideo...
2,BTA_02,Educated Affluent Homeowners,"[party_alignment: republican, media_engagement...",This segment appears to lean Republican in its...
3,BTA_03,Mainstream Working Families,[media_engagement: low],Low media engagement in this segment suggests ...
4,BTA_04,Fixed-Income Late-Life Adults,"[religiosity: none, media_engagement: low, rel...",The Fixed-Income Late-Life Adults segment pres...
5,BTA_05,Young Non-Owning Working Adults,"[religiosity: none, media_engagement: low]",This segment appears to hold little to no reli...
6,BTA_06,White Male Multi-Generation Households,"[media_engagement: low, media_engagement: high...",This segment appears to report moderate overal...


In [27]:
# Example output for one segment
row = df_bta[df_bta["segment_id"] == "BTA_04"].iloc[0]

print("Psych signals:")
print(row["psych_signals"])
print()
print("Psych summary:")
print(row["psych_summary"])

Psych signals:
['religiosity: none' 'media_engagement: low' 'religiosity: high']

Psych summary:
The Fixed-Income Late-Life Adults segment presents a notably ambiguous religious profile, with signals indicating both high religiosity and no religiosity, suggesting meaningful internal heterogeneity within this group rather than a unified attitudinal stance on faith. This contradiction may reflect a segment that is genuinely divided across a spectrum of religious engagement, or could point to subpopulations with distinct orientations that resist easy generalization. Media engagement appears low, which may indicate limited consumption of or interest in mainstream information channels.


In [28]:
### Media summary prompt
MEDIA_PROMPT = """
You are an audience intelligence analyst working on a baseline segmentation of the United States population.

Your task is to interpret the media signals associated with one audience segment and produce a short analytical summary.

The signals represent statistically meaningful media usage or information-consumption indicators associated with the segment relative to the broader population.

Your goal is to translate these signals into a clear, human-readable description of how this segment appears to consume information and interact with media environments.

Guidelines:

• Write 2–3 concise sentences.
• Use neutral analytical language (not marketing language).
• Use cautious phrasing such as "appears", "suggests", "may indicate", or "is associated with".
• Preserve all important channel cues if present.
• If signals appear mixed or contradictory, acknowledge the ambiguity rather than collapsing them.
• Do not invent behaviors not supported by the signals.
• Do not provide recommendations or messaging.

Segment information:

Segment name:
{archetype_name}

Media signals detected in the cluster:
{media_signals}

Task:

Write a concise analytical description of the media behavior of this audience segment based strictly on the signals provided.

Output only the summary text.
Do not include headings, bullet points, or explanations.
"""

In [30]:
# generate media summary
for i, row in df_bta.iterrows():

    media_text = normalize_signals(row["media_signals"])

    if media_text is None:
        df_bta.loc[i, "media_summary"] = None
        continue

    prompt = MEDIA_PROMPT.format(
        archetype_name=row["archetype_name"],
        media_signals=media_text
    )

    response = _client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )

    df_bta.loc[i, "media_summary"] = response.content[0].text.strip()

In [31]:
# check media summaries
df_bta[["segment_id", "archetype_name", "media_signals", "media_summary"]]

,segment_id,archetype_name,media_signals,media_summary
0,BTA_00,Young Diverse Working Households,"[media_usage: tiktok, media_usage: whatsapp, m...",This segment appears to rely heavily on mobile...
1,BTA_01,Older White Renters,"[media_usage: internet_frequency, media_usage:...",This segment appears to engage with digital me...
2,BTA_02,Educated Affluent Homeowners,"[media_usage: internet_frequency, media_usage:...",This segment appears to be characterized by fr...
3,BTA_03,Mainstream Working Families,"[media_usage: internet_frequency, media_usage:...",This segment appears to engage with media prim...
4,BTA_04,Fixed-Income Late-Life Adults,"[media_usage: internet_frequency, media_usage:...",This segment appears to engage with digital me...
5,BTA_05,Young Non-Owning Working Adults,"[media_usage: internet_frequency, media_usage:...",This segment appears to rely heavily on intern...
6,BTA_06,White Male Multi-Generation Households,"[media_usage: internet_frequency, media_usage:...",This segment appears to be characterized by co...


In [32]:
# Example output for one segment
row = df_bta[df_bta["segment_id"] == "BTA_04"].iloc[0]

print("Media signals:")
print(row["media_signals"])
print()
print("Media summary:")
print(row["media_summary"])

Media signals:
['media_usage: internet_frequency' 'media_usage: instagram'
 'media_usage: youtube']

Media summary:
This segment appears to engage with digital media channels despite being associated with a later life stage, suggesting meaningful internet adoption among fixed-income older adults. Signal presence across both Instagram and YouTube may indicate a degree of visual and video-oriented content consumption, spanning both short-form social browsing and longer video media. The combination of general internet frequency with platform-specific signals suggests this group is not uniformly disengaged from digital environments, though the depth and regularity of that engagement relative to the broader population remains ambiguous from these signals alone.


In [33]:
# Narrative summary prompt
NARRATIVE_PROMPT = """
You are an audience intelligence analyst working on a baseline segmentation of the United States population.

Your task is to write a concise audience narrative for one baseline segment.

The goal is to produce a human-readable analytical description that integrates structural, psychological, and media-behavior evidence into a coherent profile of the segment.

This is not marketing copy and not a persuasion message. It is an analytical segment description suitable for an audience intelligence brief.

Guidelines:

• Write 4–6 sentences in one paragraph.
• Use clear, natural, professional prose.
• Use neutral analytical language.
• Use cautious phrasing such as "appears", "suggests", "may indicate", or "is associated with".
• Ground the narrative strictly in the evidence provided.
• Preserve important ideological, attitudinal, and media-use cues if present.
• If signals are mixed or somewhat contradictory, acknowledge the ambiguity rather than forcing false coherence.
• Do not invent motivations, barriers, or recommendations.
• Do not give messaging advice.
• Do not mention "the dataset", "the cluster", or "the model".

Segment information:

Segment name:
{archetype_name}

Population share:
{population_share}

Adult population share:
{population_adult_share}

Structural profile:
{structural_profile}

Psychological signals:
{psych_signals}

Psychological summary:
{psych_summary}

Media signals:
{media_signals}

Media summary:
{media_summary}

Task:

Write one concise analytical paragraph describing this audience segment as a baseline social group within U.S. society.

Output only the paragraph text.
Do not include headings, bullet points, or explanations.
"""

In [34]:
# Generate narrative summary for each segment
for i, row in df_bta.iterrows():

    psych_text = normalize_signals(row["psych_signals"])
    media_text = normalize_signals(row["media_signals"])

    prompt = NARRATIVE_PROMPT.format(
        segment_id=row["segment_id"],
        archetype_name=row["archetype_name"],
        population_share=f"{row['population_share']:.1%}" if pd.notna(row["population_share"]) else "N/A",
        population_adult_share=f"{row['population_adult_share']:.1%}" if pd.notna(row["population_adult_share"]) else "N/A",
        structural_profile=row["structural_profile"] if pd.notna(row["structural_profile"]) else "Not available.",
        psych_signals=psych_text if psych_text is not None else "No strong psychological signals detected.",
        psych_summary=row["psych_summary"] if pd.notna(row["psych_summary"]) else "No clear psychological summary available.",
        media_signals=media_text if media_text is not None else "No strong media signals detected.",
        media_summary=row["media_summary"] if pd.notna(row["media_summary"]) else "No clear media summary available."
    )

    response = _client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )

    df_bta.loc[i, "narrative_summary"] = response.content[0].text.strip()

In [35]:
# Display the narrative summaries
df_bta[[
    "segment_id",
    "archetype_name",
    "narrative_summary"
]]

,segment_id,archetype_name,narrative_summary
0,BTA_00,Young Diverse Working Households,Young Diverse Working Households represents ap...
1,BTA_01,Older White Renters,Older White Renters represent approximately 12...
2,BTA_02,Educated Affluent Homeowners,Educated Affluent Homeowners represent approxi...
3,BTA_03,Mainstream Working Families,Mainstream Working Families represents approxi...
4,BTA_04,Fixed-Income Late-Life Adults,Fixed-Income Late-Life Adults represent approx...
5,BTA_05,Young Non-Owning Working Adults,Young Non-Owning Working Adults represent appr...
6,BTA_06,White Male Multi-Generation Households,White Male Multi-Generation Households represe...


In [36]:
# Example output for one segment
row = df_bta[df_bta["segment_id"] == "BTA_04"].iloc[0]

print("Narrative summary:")
print(row["narrative_summary"])

Narrative summary:
Fixed-Income Late-Life Adults represent approximately one in ten U.S. adults, defined structurally by retirement, advanced age (65+), lower formal educational attainment, and household incomes typically below $20,000 — a financial profile consistent with reliance on fixed sources such as Social Security or pension benefits. The segment is predominantly White and largely composed of previously married individuals, suggesting a life-stage marked by widowhood or divorce rather than active partnership. On the question of religious orientation, signals are notably contradictory, indicating both high religiosity and an absence of religiosity within the same segment; this ambiguity likely reflects genuine internal heterogeneity, with meaningfully distinct subpopulations rather than a shared attitudinal stance on faith. Despite the low media engagement signal, platform-level indicators pointing to internet use, Instagram, and YouTube suggest that at least a portion of this g

### Generate Narrative Summaries

This step synthesizes the structural, psychological, and media evidence associated with each PAS into a single analyst-readable narrative paragraph.

The narrative summary integrates multiple layers of evidence at once: structural profile, psychological signals and summaries, media signals and summaries, and segment population share. This produces a more coherent baseline description of each segment as a social group within U.S. society.

The resulting `narrative_summary` serves as the core descriptive text of the BTA card and will later support downstream retrieval, audience selection, and strategy-generation workflows.

In [37]:
pd.set_option("display.max_columns", None)
df_bta.head()

,segment_id,cluster_id,archetype_name,baseline_layer_version,population_share,population_adult_share,population_rank,dominant_age_bin,dominant_sex_label,dominant_race_eth,dominant_edu_tier,dominant_emp_tier,dominant_household_income_tier,dominant_mar_tier,dominant_tenure,structural_profile,psych_signals,psych_summary,media_signals,media_summary,motivational_drivers,key_barriers,trust_cues,channel_implications,messaging_implications,offer_implications,susceptibility_notes,narrative_summary,tags,source_layers,rag_text
0,BTA_00,0,Young Diverse Working Households,v1,0.055954,0.056,None,35-44,Male,Hispanic,HS_or_less,Employed,100-199k,Married,Owner,Segment composed of individuals primarily aged...,"[media_engagement: low, media_engagement: high...",This segment shows mixed signals around media ...,"[media_usage: tiktok, media_usage: whatsapp, m...",This segment appears to rely heavily on mobile...,None,None,None,None,None,None,None,Young Diverse Working Households represents ap...,None,None,None
1,BTA_01,1,Older White Renters,v1,0.123532,0.124,None,65+,Female,White,HS_or_less,Retired,50-99k,Married,Renter,Segment composed of individuals primarily aged...,"[media_engagement: low, religiosity: none, ide...",This segment appears to hold conservative ideo...,"[media_usage: internet_frequency, media_usage:...",This segment appears to engage with digital me...,None,None,None,None,None,None,None,Older White Renters represent approximately 12...,None,None,None
2,BTA_02,2,Educated Affluent Homeowners,v1,0.183602,0.184,None,55-64,Female,White,Bachelor,Employed,100-199k,Married,Owner,Segment composed of individuals primarily aged...,"[party_alignment: republican, media_engagement...",This segment appears to lean Republican in its...,"[media_usage: internet_frequency, media_usage:...",This segment appears to be characterized by fr...,None,None,None,None,None,None,None,Educated Affluent Homeowners represent approxi...,None,None,None
3,BTA_03,3,Mainstream Working Families,v1,0.190217,0.190,None,35-44,Female,White,HS_or_less,Employed,50-99k,Married,Owner,Segment composed of individuals primarily aged...,[media_engagement: low],Low media engagement in this segment suggests ...,"[media_usage: internet_frequency, media_usage:...",This segment appears to engage with media prim...,None,None,None,None,None,None,None,Mainstream Working Families represents approxi...,None,None,None
4,BTA_04,4,Fixed-Income Late-Life Adults,v1,0.101945,0.102,None,65+,Female,White,HS_or_less,Retired,0-19k,Previously_Married,No_Rent,Segment composed of individuals primarily aged...,"[religiosity: none, media_engagement: low, rel...",The Fixed-Income Late-Life Adults segment pres...,"[media_usage: internet_frequency, media_usage:...",This segment appears to engage with digital me...,None,None,None,None,None,None,None,Fixed-Income Late-Life Adults represent approx...,None,None,None


### Populate Source Layers

This step records the baseline data layers used to construct each BTA card.

At this stage, all segments are derived from the same integrated baseline stack:

- **ACS** for structural and demographic attributes
- **GSS** for psychological and attitudinal signals
- **Pew / NPORS** for media behavior signals

The `source_layers` field preserves provenance and makes the BTA cards easier to audit, filter, and reuse in later retrieval and strategy-generation workflows.

In [38]:
# adding source layers info
df_bta["source_layers"] = [
    ["ACS_structural", "GSS_psych", "Pew_NPORS_media"]
    for _ in range(len(df_bta))
]

In [39]:
# quick check
df_bta[["segment_id", "archetype_name", "source_layers"]].head()

,segment_id,archetype_name,source_layers
0,BTA_00,Young Diverse Working Households,"[ACS_structural, GSS_psych, Pew_NPORS_media]"
1,BTA_01,Older White Renters,"[ACS_structural, GSS_psych, Pew_NPORS_media]"
2,BTA_02,Educated Affluent Homeowners,"[ACS_structural, GSS_psych, Pew_NPORS_media]"
3,BTA_03,Mainstream Working Families,"[ACS_structural, GSS_psych, Pew_NPORS_media]"
4,BTA_04,Fixed-Income Late-Life Adults,"[ACS_structural, GSS_psych, Pew_NPORS_media]"


### Generate PTA Tags

This step creates a compact set of machine-readable tags for each BTA card.

The tags are derived from dominant structural attributes and the available psychological and media signals. Their purpose is not to replace the narrative summary, but to provide a lightweight indexing layer that supports filtering, retrieval, and later segment selection workflows.

At this stage, tags remain descriptive and baseline-oriented. They do not encode objective-specific judgments or campaign recommendations.

In [40]:
# Tags creating function
def normalize_signals_to_list(x):
    if x is None:
        return []
    if isinstance(x, float) and pd.isna(x):
        return []
    if isinstance(x, np.ndarray):
        x = x.tolist()
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    x = str(x).strip()
    return [x] if x else []


def make_bta_tags(row):
    tags = []

    # structural tags
    structural_fields = [
        "dominant_age_bin",
        "dominant_sex_label",
        "dominant_race_eth",
        "dominant_edu_tier",
        "dominant_emp_tier",
        "dominant_household_income_tier",
        "dominant_mar_tier",
        "dominant_tenure",
    ]

    for col in structural_fields:
        val = row.get(col)
        if pd.notna(val):
            tags.append(str(val).lower())

    # psych tags
    psych_tags = normalize_signals_to_list(row.get("psych_signals"))
    tags.extend([t.lower().replace(": ", "_").replace(":", "_").replace(" ", "_") for t in psych_tags])

    # media tags
    media_tags = normalize_signals_to_list(row.get("media_signals"))
    tags.extend([t.lower().replace(": ", "_").replace(":", "_").replace(" ", "_") for t in media_tags])

    # remove duplicates while preserving order
    seen = set()
    clean_tags = []
    for tag in tags:
        if tag not in seen:
            seen.add(tag)
            clean_tags.append(tag)

    return clean_tags

In [41]:
# apply the function to create tags for each segment
df_bta["tags"] = df_bta.apply(make_bta_tags, axis=1)

In [42]:
# quick check
df_bta[["segment_id", "archetype_name", "tags"]].head()

,segment_id,archetype_name,tags
0,BTA_00,Young Diverse Working Households,"[35-44, male, hispanic, hs_or_less, employed, ..."
1,BTA_01,Older White Renters,"[65+, female, white, hs_or_less, retired, 50-9..."
2,BTA_02,Educated Affluent Homeowners,"[55-64, female, white, bachelor, employed, 100..."
3,BTA_03,Mainstream Working Families,"[35-44, female, white, hs_or_less, employed, 5..."
4,BTA_04,Fixed-Income Late-Life Adults,"[65+, female, white, hs_or_less, retired, 0-19..."


### Generate RAG Text

This step creates a retrieval-ready text representation of each BTA card.

The `rag_text` field condenses the most important descriptive elements of the segment into a single structured paragraph, combining identity, population relevance, structural profile, psychological interpretation, media interpretation, and the overall narrative summary.

This field is designed for downstream retrieval workflows, enabling later components of Market Kinetics to search, compare, and reason over BTA cards as reusable audience intelligence objects.

In [43]:
RAG_INTERPRETATION_PROMPT = """
You are an audience intelligence analyst.

Your task is to write a short retrieval-oriented interpretation of one audience segment for use in a retrieval-augmented generation system.

The goal is not to restate all details already provided. Instead, produce a compact analytical synthesis that captures the segment’s overall character in 1–2 sentences.

Guidelines:
- Write exactly 1–2 sentences.
- Use neutral analytical language.
- Use cautious phrasing such as "appears", "suggests", or "may indicate".
- Focus on the segment’s overall character and any notable tensions or heterogeneity.
- Do not repeat population size, structural details, or platform names unless necessary.
- Do not invent unsupported motivations, recommendations, or messaging advice.
- Output only the interpretation text.

Segment name:
{archetype_name}

Structural profile:
{structural_profile}

Psychological profile:
{psych_summary}

Media behavior:
{media_summary}
"""

In [44]:
# helper function to generate RAG interpretation text for each segment
def generate_rag_interpretation(row):
    prompt = RAG_INTERPRETATION_PROMPT.format(
        archetype_name=row["archetype_name"],
        structural_profile=row["structural_profile"] if pd.notna(row["structural_profile"]) else "Not available.",
        psych_summary=row["psych_summary"] if pd.notna(row["psych_summary"]) else "Not available.",
        media_summary=row["media_summary"] if pd.notna(row["media_summary"]) else "Not available."
    )

    response = _client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=120,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.content[0].text.strip()

In [45]:
# function to generate RAG text for each segment

def build_rag_text(row):
    parts = []

    # identity
    parts.append(f"Segment ID: {row['segment_id']}")
    parts.append(f"Segment name: {row['archetype_name']}")

    # population relevance
    pop_bits = []
    if pd.notna(row.get("population_share")):
        pop_bits.append(f"{row['population_share']:.1%} of the U.S. population")
    if pd.notna(row.get("population_adult_share")):
        pop_bits.append(f"{row['population_adult_share']:.1%} of the adult population")
    if pop_bits:
        parts.append("Population relevance: " + "; ".join(pop_bits))

    # structural
    if pd.notna(row.get("structural_profile")) and row["structural_profile"]:
        parts.append(f"Structural profile: {str(row['structural_profile']).strip()}")

    # psych
    if pd.notna(row.get("psych_summary")) and row["psych_summary"]:
        parts.append(f"Psychological profile: {str(row['psych_summary']).strip()}")

    # media
    if pd.notna(row.get("media_summary")) and row["media_summary"]:
        parts.append(f"Media behavior: {str(row['media_summary']).strip()}")

    # short LLM interpretation
    interpretation = generate_rag_interpretation(row)
    parts.append(f"Interpretation: {interpretation}")

    return "\n".join(parts)

In [46]:
# apply the function to create RAG text for each segment
df_bta["rag_text"] = df_bta.apply(build_rag_text, axis=1)

In [47]:
# quick check
row = df_bta.iloc[4]

print(row["rag_text"])

Segment ID: BTA_04
Segment name: Fixed-Income Late-Life Adults
Population relevance: 10.2% of the U.S. population; 10.2% of the adult population
Structural profile: Segment composed of individuals primarily aged 65+, retired, previously married, with hs or less education, household income typically in the 0-19k range, mostly non-paying occupants, predominantly White.
Psychological profile: The Fixed-Income Late-Life Adults segment presents a notably ambiguous religious profile, with signals indicating both high religiosity and no religiosity, suggesting meaningful internal heterogeneity within this group rather than a unified attitudinal stance on faith. This contradiction may reflect a segment that is genuinely divided across a spectrum of religious engagement, or could point to subpopulations with distinct orientations that resist easy generalization. Media engagement appears low, which may indicate limited consumption of or interest in mainstream information channels.
Media behavior

### BTA Dataset Quality Check

This step performs a basic integrity check on the BTA dataset before export.

The check verifies that all segments contain the required analytical fields:
- structural profile
- psychological summary
- media summary
- narrative summary
- rag text representation

Any missing values are flagged so they can be reviewed before finalizing the baseline BTA layer.

In [48]:
# columns that must be populated
required_cols = [
    "structural_profile",
    "psych_summary",
    "media_summary",
    "narrative_summary",
    "rag_text"
]

# check for missing values
qa_missing = df_bta[required_cols].isnull().sum()

print("Missing values by column:")
print(qa_missing)

Missing values by column:
structural_profile    0
psych_summary         0
media_summary         0
narrative_summary     0
rag_text              0
dtype: int64


In [49]:
# Export BTA cards baseline table
bta_path = OUTPUT_DIR / "mk_bta_baseline.parquet"
df_bta.to_parquet(bta_path, index=False)

print("✓ Saved:", bta_path)

✓ Saved: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_processed/bta_cards/mk_bta_baseline.parquet


In [50]:
# Export BTA cards baseline table — CSV
csv_path = OUTPUT_DIR / "mk_bta_baseline.csv"
df_bta.to_csv(csv_path, index=False)

print("✓ Saved:", csv_path)

✓ Saved: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_processed/bta_cards/mk_bta_baseline.csv


In [51]:
# Export RAG corpus
import json

rag_path = OUTPUT_DIR / "mk_bta_rag_corpus.jsonl"

with open(rag_path, "w") as f:
    for _, row in df_bta.iterrows():
        record = {
            "segment_id":    row["segment_id"],
            "archetype_name": row["archetype_name"],
            "tags":          row["tags"],
            "rag_text":      row["rag_text"]
        }
        f.write(json.dumps(record) + "\n")

print("✓ Saved:", rag_path)
print(f"  Records: {len(df_bta)}")

✓ Saved: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_processed/bta_cards/mk_bta_rag_corpus.jsonl
  Records: 7


### Notebook Summary

This notebook converted baseline societal clusters into structured
**Baseline Target Audience (BTA) cards**, integrating structural
(ACS), psychological (GSS), and media behavior (NPORS/Pew) signals.

The resulting dataset represents the **Market Kinetics baseline
audience intelligence layer**, providing both analyst-readable segment
profiles and RAG-ready representations for downstream retrieval,
targeting, and future Target Audience Analysis Worksheet (TAAW)
generation.

**Outputs saved to** `data/societal_processed/bta_cards/`:
- `mk_bta_baseline.parquet` — full BTA card table
- `mk_bta_baseline.csv` — tabular export for inspection
- `mk_bta_rag_corpus.jsonl` — retrieval corpus for embedding
  and vector search pipelines